# Advanced: Retrain with Class Weights + SMOTE
## Comprehensive Solution for Class Imbalance

This uses both:
- **SMOTE**: Oversamples minority classes to 10% of BENIGN
- **Class Weights**: Further penalizes minority class errors
- **Both base learners**: LightGBM + Bagging with balanced weights

In [ ]:
# Install imbalanced-learn for SMOTE
import subprocess
subprocess.check_call(['.venv\\Scripts\\pip.exe', 'install', '-q', 'imbalanced-learn'])
print("SMOTE library installed!")

In [ ]:
# IMPORT LIBRARIES

import sys, os, glob, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc as sk_auc
)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

print("="*80)
print("RETRAINING WITH CLASS WEIGHTS + SMOTE (ADVANCED)")
print("="*80)

In [ ]:
# CONFIG

CSV_FOLDER   = './MachineLearningCVE'  
LABEL_COL    = 'Label'
N_FOLDS      = 10
TEST_SIZE    = 0.30        
RANDOM_STATE = 42
OUTPUT_DIR   = '.'
os.makedirs(OUTPUT_DIR, exist_ok=True)

import sklearn
print(f'Scikit-learn : {sklearn.__version__}')
print(f'LightGBM     : {lgb.__version__}')
print(f'Pandas       : {pd.__version__}')

In [ ]:
# LOAD DATA

csv_files = sorted(glob.glob(os.path.join(CSV_FOLDER, '*.csv')))
if not csv_files:
    raise FileNotFoundError(f'No CSV files in {CSV_FOLDER}')
dfs = []
for f in csv_files:
    print(f'  Loading {os.path.basename(f)}')
    dfs.append(pd.read_csv(f, low_memory=False, encoding='latin-1'))
df = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(df):,}')
del dfs; gc.collect()

In [ ]:
# CLEAN DATA

df.columns = df.columns.str.strip()

# Drop irrelevant ID columns
drop_cols = [c for c in ['Flow ID','Source IP','Destination IP','Timestamp',
                          'Src IP','Dst IP','src_ip','dst_ip'] if c in df.columns]
if drop_cols:
    df.drop(columns=drop_cols, inplace=True)

# Replace infinities with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop COLUMNS with >50% missing
df.dropna(axis=1, thresh=int(len(df) * 0.5), inplace=True)  
df.dropna(axis=0, inplace=True)                               

# Deduplicate
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Dedup removed {before - len(df):,} rows')
print(f'Shape after dedup: {df.shape}')

In [ ]:
# ENCODE & SCALE

le = LabelEncoder()
df['label_enc'] = le.fit_transform(df[LABEL_COL].astype(str))
class_names = le.classes_
n_classes   = len(class_names)
print(f'\nClasses ({n_classes}):')
for i, n in enumerate(class_names): 
    print(f'  {i:2d}: {n}')

X = df.drop(columns=[LABEL_COL, 'label_enc']).select_dtypes(include=[np.number])
y = df['label_enc']

# 70/30 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'\nTrain: {X_train.shape} | Test: {X_test.shape}')

scaler  = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

del df; gc.collect()
print('Encoding & scaling done')

In [ ]:
# APPLY SMOTE FOR CLASS BALANCING
print("\nApplying SMOTE (Synthetic Minority Oversampling)...")

# SMOTE: oversample minority classes to 10% of majority class
smote = SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"Before SMOTE:  {X_train_scaled.shape}")
print(f"After SMOTE:   {X_train_smote.shape}")

# Show new class distribution
unique, counts = np.unique(y_train_smote, return_counts=True)
print("\nNew class distribution (after SMOTE):")
for u, c in zip(unique, counts):
    print(f"  {class_names[u]:30s}: {c:6,} samples")

In [ ]:
# MODEL PARAMETERS WITH CLASS WEIGHTS + SMOTE
print("\n" + "="*80)
print("MODEL CONFIGURATION (WITH CLASS WEIGHTS + SMOTE)")
print("="*80)

# LightGBM with class weights
lgbm_params = dict(
    objective         = 'multiclass',
    metric            = 'multi_logloss',
    num_class         = n_classes,
    n_estimators      = 100,          
    learning_rate     = 0.05,
    max_depth         = 8,
    num_leaves        = 63,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    is_unbalance      = False,      # Disabled: SMOTE already balanced
    class_weight      = 'balanced',  # Keep weights as safeguard
    device            = 'cpu',
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbose           = -1,
)

# Bagging + DT with class weights
n_features = X_train_scaled.shape[1]
bagging_params = dict(
    estimator   = DecisionTreeClassifier(
                      criterion     = 'gini',
                      max_depth     = 10,
                      min_samples_split = 20,
                      min_samples_leaf  = 10,
                      class_weight  = 'balanced',  
                      random_state  = RANDOM_STATE
                  ),
    n_estimators = 50,
    max_samples  = 0.8,
    max_features = max(1, int(np.sqrt(n_features))), 
    bootstrap    = True,
    n_jobs       = -1,
    random_state = RANDOM_STATE,
)

print(f'✓ SMOTE: Oversampled minority to 10% of BENIGN')
print(f'✓ LightGBM: class_weight="balanced"')
print(f'✓ Bagging: DecisionTree with class_weight="balanced"')
print(f'✓ Meta-learner: LogisticRegression with class_weight="balanced"')

In [ ]:
# CROSS-VALIDATION WITH STACKING ON SMOTE DATA
print(f'\nRunning {N_FOLDS}-Fold CV (on SMOTE balanced data)...\n')
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

Xa  = X_train_smote.values
ya  = y_train_smote.values
Xta = X_test_scaled.values

oof_lgbm    = np.zeros((len(Xa), n_classes))
oof_bagging = np.zeros((len(Xa), n_classes))
test_lgbm_accum    = np.zeros((len(Xta), n_classes))
test_bagging_accum = np.zeros((len(Xta), n_classes))

fold_results   = []   
fold_loss_curves = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(Xa, ya), 1):
    print(f'  Fold {fold}/{N_FOLDS}', end='')
    Xtr, Xval = Xa[tr_idx], Xa[val_idx]
    ytr, yval = ya[tr_idx], ya[val_idx]

    # LightGBM 
    lgbm_fold = lgb.LGBMClassifier(**lgbm_params)
    evals = {}
    lgbm_fold.fit(
        Xtr, ytr,
        eval_set       = [(Xtr, ytr), (Xval, yval)],
        eval_names     = ['train', 'eval'],
        callbacks      = [
            lgb.record_evaluation(evals),
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(-1),
        ],
    )
    oof_lgbm[val_idx]   = lgbm_fold.predict_proba(Xval)
    test_lgbm_accum    += lgbm_fold.predict_proba(Xta) 

    train_loss = evals['train']['multi_logloss']
    val_loss   = evals['eval']['multi_logloss']
    best_iter  = lgbm_fold.best_iteration_
    fold_loss_curves.append({
        'fold': fold, 'train': train_loss, 'eval': val_loss, 'best': best_iter
    })
    print(f' | LGBM OK (best={best_iter})', end='')

    # Bagging + DT 
    bagging_fold = BaggingClassifier(**bagging_params)
    bagging_fold.fit(Xtr, ytr)
    oof_bagging[val_idx]   = bagging_fold.predict_proba(Xval)
    test_bagging_accum    += bagging_fold.predict_proba(Xta)
    print(' | Bagging OK')

    oof_val_pred_lgbm    = np.argmax(oof_lgbm[val_idx], axis=1)
    oof_val_pred_bagging = np.argmax(oof_bagging[val_idx], axis=1)

    fold_results.append({
        'fold'           : fold,
        'val_loss_lgbm'  : val_loss[best_iter - 1] if best_iter else val_loss[-1],
        'lgbm_acc'       : accuracy_score(yval, oof_val_pred_lgbm),
        'lgbm_f1'        : f1_score(yval, oof_val_pred_lgbm, average='weighted', zero_division=0),
        'bagging_acc'    : accuracy_score(yval, oof_val_pred_bagging),
        'bagging_f1'     : f1_score(yval, oof_val_pred_bagging, average='weighted', zero_division=0),
    })
    gc.collect()

# Average test predictions across all folds
test_lgbm_avg    = test_lgbm_accum    / N_FOLDS
test_bagging_avg = test_bagging_accum / N_FOLDS

print(f'\nOOF shapes: lgbm={oof_lgbm.shape}, bagging={oof_bagging.shape}')
print('Cross-validation complete')

In [ ]:
# META-LEARNER WITH CLASS WEIGHTS
M_train = np.hstack([oof_lgbm, oof_bagging])
M_test  = np.hstack([test_lgbm_avg, test_bagging_avg])
print(f'Meta-feature matrix: Train {M_train.shape} | Test {M_test.shape}')

meta = LogisticRegression(
    max_iter     = 1000,
    random_state = RANDOM_STATE,
    class_weight = 'balanced',
    solver       = 'lbfgs',
    n_jobs       = -1,
)
meta.fit(M_train, ya)
print('Meta-learner trained')

In [ ]:
# Save models
SAVE_DIR = './saved_model'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(meta,          f'{SAVE_DIR}/meta_learner.pkl')
joblib.dump(scaler,        f'{SAVE_DIR}/scaler.pkl')
joblib.dump(le,            f'{SAVE_DIR}/label_encoder.pkl')
joblib.dump(fold_results,  f'{SAVE_DIR}/fold_results.pkl')

print(f'✅ Models saved to {SAVE_DIR}/')

In [ ]:
# FINAL EVALUATION ON TEST SET
y_pred     = meta.predict(M_test)
y_prob     = meta.predict_proba(M_test)

acc        = accuracy_score(y_test, y_pred)
prec       = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec        = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1         = f1_score(y_test, y_pred, average='weighted', zero_division=0)
ll         = log_loss(y_test, y_prob)
auc_score  = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

print('\n' + '=' * 50)
print('  STACKED ENSEMBLE — TEST SET METRICS')
print('  (Trained with SMOTE + Class Weights)')
print('=' * 50)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  AUC-ROC   : {auc_score:.4f}')
print(f'  Log Loss  : {ll:.4f}')
print('=' * 50)

In [ ]:
# Detailed classification report
print('\n' + '=' * 80)
print('PER-CLASS PERFORMANCE')
print('=' * 80)
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0, digits=4))

In [ ]:
print("\n" + "="*80)
print("✅ ADVANCED RETRAINING COMPLETE!")
print("="*80)
print("\nImprovements made:")
print("  • SMOTE: Oversampled minority classes to 10% of BENIGN")
print("  • LightGBM: Added class_weight='balanced'")
print("  • Meta-learner: LogisticRegression with class_weight='balanced'")
print("\nExpected improvements:")
print("  • Much better detection of minority classes")
print("  • DDoS should no longer be confused with Infiltration")
print("  • PortScan should no longer be confused with BENIGN")
print("  • Significant improvement in F1-scores for rare attacks")
print("\nModels saved to ./saved_model/")
print("Test the model with your Flask backend!")
print("="*80)